In [1]:
import json
import sqlite3
import random
import time
from datetime import datetime
from queue import Queue
from threading import Thread

In [2]:
def read_json(file):
    try:
        with open(file, 'r') as f:
            return json.load(f)
    except Exception as e:
        print("Error reading file:", e)
        return []


In [3]:
def validate_order(order):
    try:
        required = ["order_id", "user", "city", "amount", "order_time"]
        for key in required:
            if key not in order or order[key] is None:
                return False

        if not isinstance(order["amount"], int):
            return False

        return True
    except:
        return False

In [4]:
def clean_data(orders):
    clean_orders = []
    for o in orders:
        if validate_order(o):
            clean_orders.append(o)
        else:
            print("Corrupt data skipped:", o)
    return clean_orders

In [5]:
def add_features(order):
    try:
        order["delivery_time"] = random.randint(10, 45)
        order["distance"] = random.randint(1, 15)
        order["driver_id"] = random.randint(1, 10)
        return order
    except:
        return None

In [6]:
def process_with_retry(order, retries=3):
    for i in range(retries):
        try:
            return add_features(order)
        except Exception as e:
            print(f"Retry {i+1} failed:", e)
            time.sleep(1)
    return None

In [7]:
def setup_db():
    conn = sqlite3.connect("food_delivery.db", check_same_thread=False)
    cursor = conn.cursor()

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS orders (
        order_id INTEGER,
        user TEXT,
        city TEXT,
        amount INTEGER,
        delivery_time INTEGER,
        distance INTEGER,
        driver_id INTEGER
    )
    """)

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS drivers (
        driver_id INTEGER,
        name TEXT,
        city TEXT
    )
    """)

    conn.commit()
    return conn


In [8]:
def insert_orders(conn, orders):
    cursor = conn.cursor()
    for o in orders:
        cursor.execute("""
        INSERT INTO orders VALUES (?, ?, ?, ?, ?, ?, ?)
        """, (
            o["order_id"], o["user"], o["city"], o["amount"],
            o["delivery_time"], o["distance"], o["driver_id"]
        ))
    conn.commit()

In [9]:
def insert_drivers(conn, drivers):
    cursor = conn.cursor()
    for d in drivers:
        cursor.execute("INSERT INTO drivers VALUES (?, ?, ?)",
                       (d["driver_id"], d["name"], d["city"]))
    conn.commit()

In [21]:
def run_analytics(conn):
    cursor = conn.cursor()

    print(" --- Analytics ---\n")

    #Total--orders
    cursor.execute("SELECT COUNT(*) FROM orders")
    total_orders = cursor.fetchone()[0]

    # orders per city
    cursor.execute("SELECT city, COUNT(*) FROM orders GROUP BY city")
    city_data = cursor.fetchall()

    # Top-driver
    cursor.execute("""
    SELECT driver_id, COUNT(*) as total
    FROM orders
    GROUP BY driver_id
    ORDER BY total DESC
    LIMIT 1
    """)
    top_driver = cursor.fetchone()

    # Avg delivery time
    cursor.execute("SELECT AVG(delivery_time) FROM orders")
    avg_time = cursor.fetchone()[0]

    print("Total Orders:", total_orders)
    print("Orders per City:", city_data)
    print("Top Driver:", top_driver)
    print(f"Avg Delivery Time: {avg_time:.2f} mins")
    print("\n--------------------------\n")

    return total_orders, top_driver, avg_time

In [22]:
queue = Queue()

def generate_live_order(order_id):
    return {
        "order_id": order_id,
        "user": random.choice(["Arun", "Priya", "Rahul"]),
        "city": random.choice(["Chennai", "Bangalore"]),
        "amount": random.randint(100, 500),
        "order_time": str(datetime.now())
    }

In [23]:
def producer():
    for i in range(1000, 1010):
        order = generate_live_order(i)
        print("New Order:", order,"\n")
        queue.put(order)
        time.sleep(2)

In [24]:
def consumer(conn):
    while True:
        if not queue.empty():
            order = queue.get()

            if not validate_order(order):
                print("Invalid streaming order skipped")
                continue

            processed = process_with_retry(order)

            if processed:
                insert_orders(conn, [processed])
                print("Processed & Stored:", processed,"\n")

                # Live analytics update
                run_analytics(conn)

In [25]:
if __name__ == "__main__":

    orders = read_json("orders.json")
    drivers = read_json("drivers.json")

    clean_orders = clean_data(orders)

    processed_orders = []
    for order in clean_orders:
        processed = process_with_retry(order)
        if processed:
            processed_orders.append(processed)

    conn = setup_db()

    insert_orders(conn, processed_orders)
    insert_drivers(conn, drivers)

    total_orders, top_driver, avg_time = run_analytics(conn)

    print("\n........FINAL OUTPUT........")
    print(f"Total Orders: {total_orders}")
    print(f"Top Driver: {top_driver}")
    print(f"Avg Delivery Time: {avg_time:.2f} mins ")
    print("\n...........................\n")

    print("Streaming Started...\n")
    Thread(target=producer).start()
    Thread(target=consumer, args=(conn,), daemon=True).start()

    time.sleep(15)


Corrupt data skipped: {'order_id': 31, 'user': None, 'city': 'Chennai', 'amount': 250, 'order_time': '2026-03-30 11:15:00'}
Corrupt data skipped: {'order_id': 32, 'user': 'Vijay', 'city': None, 'amount': 300, 'order_time': '2026-03-30 11:18:00'}
Corrupt data skipped: {'order_id': 33, 'user': 'Anu', 'city': 'Mumbai', 'amount': 'invalid', 'order_time': '2026-03-30 11:20:00'}
Corrupt data skipped: {'order_id': 34, 'user': 'Arun', 'city': 'Chennai', 'amount': None, 'order_time': '2026-03-30 11:22:00'}
 --- Analytics ---

Total Orders: 154
Orders per City: [('Bangalore', 52), ('Chennai', 46), ('Hyderabad', 28), ('Mumbai', 28)]
Top Driver: (2, 21)
Avg Delivery Time: 25.71 mins

--------------------------


........FINAL OUTPUT........
Total Orders: 154
Top Driver: (2, 21)
Avg Delivery Time: 25.71 mins 

...........................

Streaming Started...

New Order: {'order_id': 1000, 'user': 'Priya', 'city': 'Chennai', 'amount': 286, 'order_time': '2026-03-30 08:49:34.352988'} 

Processed & S